In [2]:
# Cell 1: Imports
import pandas as pd
import numpy as np
import requests
import joblib
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Cell 2: Configurations & Loaders
LATITUDE = 21.0050
LONGITUDE = 106.8333
MODEL_PATH = 'ensemble_model_pm25.pkl'
SCALER_PATH = 'scaler.pkl' # Giả sử bạn đã lưu scaler

WEATHER_API_URL = "https://api.open-meteo.com/v1/forecast"
AIR_QUALITY_API_URL = "https://air-quality-api.open-meteo.com/v1/air-quality"
BASE_MET_FEATURES = ['temperature_2m', 'relative_humidity_2m', 'precipitation', 'wind_u', 'wind_v']

# Tải mô hình
model = joblib.load(MODEL_PATH)
scaler = joblib.load(SCALER_PATH)

# Lấy feature_names từ mô hình (nếu có lưu) hoặc tự định nghĩa lại 23 cột
feature_names = model.feature_names_in_ if hasattr(model, 'feature_names_in_') else []

# Cell 3: Fetch Data & Preprocess
def evaluate_yesterday():
    print("BẮT ĐẦU ĐÁNH GIÁ HIỆU SUẤT AI TRONG 24 GIỜ QUA...")
    
    # Lấy dữ liệu 2 ngày qua (để tính được rolling 24h cho ngày hôm qua)
    w_params = {
        "latitude": LATITUDE, "longitude": LONGITUDE, 
        "hourly": "temperature_2m,relative_humidity_2m,precipitation,wind_speed_10m,wind_direction_10m", 
        "timezone": "Asia/Bangkok", "past_days": 2, "forecast_days": 0
    }
    aq_params = {
        "latitude": LATITUDE, "longitude": LONGITUDE, 
        "hourly": "pm2_5,nitrogen_dioxide,sulphur_dioxide,carbon_monoxide,ozone,aerosol_optical_depth", 
        "timezone": "Asia/Bangkok", "past_days": 2, "forecast_days": 0
    }
    
    # Kéo dữ liệu
    df_w = pd.DataFrame(requests.get(WEATHER_API_URL, params=w_params).json()['hourly'])
    df_aq = pd.DataFrame(requests.get(AIR_QUALITY_API_URL, params=aq_params).json()['hourly'])
    df_eval = pd.merge(df_w, df_aq, on='time')
    df_eval['time'] = pd.to_datetime(df_eval['time'])
    
    # Tiền xử lý (Giống hệt app.py)
    df_eval['hour'] = df_eval['time'].dt.hour
    df_eval['day_of_week'] = df_eval['time'].dt.dayofweek
    df_eval['month'] = df_eval['time'].dt.month
    
    wind_dir_rad = np.radians(df_eval['wind_direction_10m'])
    df_eval['wind_u'] = -df_eval['wind_speed_10m'] * np.sin(wind_dir_rad)
    df_eval['wind_v'] = -df_eval['wind_speed_10m'] * np.cos(wind_dir_rad)
    
    # Trích xuất Rolling
    for feature in BASE_MET_FEATURES:
        df_eval[f'{feature}_rolling_12h'] = df_eval[feature].rolling(window=12, min_periods=1).mean()
        df_eval[f'{feature}_rolling_24h'] = df_eval[feature].rolling(window=24, min_periods=1).mean()
        
    # Lọc ĐÚNG 24 giờ của ngày hôm qua
    yesterday = datetime.now().date() - timedelta(days=1)
    df_yesterday = df_eval[df_eval['time'].dt.date == yesterday].reset_index(drop=True)
    
    if len(df_yesterday) == 0:
        print("Lỗi: Không lấy được dữ liệu ngày hôm qua.")
        return
        
    # Chuẩn hóa và Dự báo
    X_unscaled = df_yesterday[feature_names]
    X_scaled = scaler.transform(X_unscaled)
    
    df_yesterday['AI_Predicted_PM25'] = model.predict(X_scaled)
    df_yesterday['Actual_PM25'] = df_yesterday['pm2_5']
    
    # Tính sai số
    mae = mean_absolute_error(df_yesterday['Actual_PM25'], df_yesterday['AI_Predicted_PM25'])
    rmse = np.sqrt(mean_squared_error(df_yesterday['Actual_PM25'], df_yesterday['AI_Predicted_PM25']))
    
    print(f"--- KẾT QUẢ ĐÁNH GIÁ NGÀY {yesterday.strftime('%d/%m/%Y')} ---")
    print(f"1. MAE  : {mae:.2f} µg/m³")
    print(f"2. RMSE : {rmse:.2f} µg/m³")
    
    # Vẽ biểu đồ so sánh trực quan
    plt.figure(figsize=(12, 5))
    plt.plot(df_yesterday['time'], df_yesterday['Actual_PM25'], label='Thực tế (API CAMS)', color='blue', marker='o')
    plt.plot(df_yesterday['time'], df_yesterday['AI_Predicted_PM25'], label='AI Dự báo', color='red', linestyle='--', marker='x')
    plt.title(f"Đánh giá hiệu suất AI - Ngày {yesterday.strftime('%d/%m/%Y')}", fontsize=14)
    plt.xlabel("Giờ trong ngày")
    plt.ylabel("Nồng độ PM2.5 (µg/m³)")
    plt.legend()
    plt.grid(True, linestyle=':', alpha=0.7)
    plt.tight_layout()
    plt.savefig('daily_evaluation_chart.png')
    plt.show()

# Cell 4: Run
if __name__ == "__main__":
    evaluate_yesterday()

FileNotFoundError: [Errno 2] No such file or directory: 'scaler.pkl'